In [1]:
print("hi")

hi


In [2]:
import numpy as np
import pandas as pd
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from scipy.stats import ks_2samp
from xgboost import XGBRegressor

In [6]:
train_df = pd.read_csv("train.csv")
production_df = pd.read_csv("test.csv")
production_df.head(10)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1461,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
1,1462,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
2,1463,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
3,1464,60,RL,78.0,9978,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
4,1465,120,RL,43.0,5005,Pave,NaN,IR1,HLS,AllPub,...,144,0,NaN,NaN,NaN,0,1,2010,WD,Normal
5,1466,60,RL,75.0,10000,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,4,2010,WD,Normal
6,1467,20,RL,NaN,7980,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,GdPrv,Shed,500,3,2010,WD,Normal
7,1468,60,RL,63.0,8402,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,5,2010,WD,Normal
8,1469,20,RL,85.0,10176,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,2,2010,WD,Normal
9,1470,20,RL,70.0,8400,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,4,2010,WD,Normal


In [7]:
TARGET_COLUMN = "SalePrice"
ID_COLUMN = "Id"
RANDOM_STATE = 42
ID_COLUMN = "Id"
production_features = production_df.drop(columns=[ID_COLUMN], errors="ignore")

In [9]:
X = train_df.drop(columns=[TARGET_COLUMN])

if ID_COLUMN in X.columns:
    X = X.drop(columns=[ID_COLUMN])

y = train_df[TARGET_COLUMN]
production_features = production_df.drop(columns=[ID_COLUMN], errors="ignore")

In [10]:
def build_preprocessor(X):
    numeric_columns = X.select_dtypes(include=["int64", "float64"]).columns
    categorical_columns = X.select_dtypes(include=["object", "string"]).columns

    numeric_preprocessor = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ]
    )

    categorical_preprocessor = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_preprocessor, numeric_columns),
            ("cat", categorical_preprocessor, categorical_columns),
        ]
    )

    return preprocessor

In [11]:
X_train, X_valid, y_train, y_valid = train_test_split(X,y,test_size=0.20,random_state=RANDOM_STATE,)

In [12]:
model = Pipeline(
    steps=[
        ("preprocessor", build_preprocessor(X_train)),
        (
            "model",
            XGBRegressor(
                n_estimators=300,
                learning_rate=0.05,
                max_depth=4,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="reg:squarederror",
                random_state=42,
            ),
        ),
    ]
)

model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [13]:
production_predictions = model.predict(production_features)

if ID_COLUMN in production_df.columns:
    production_output = production_df[[ID_COLUMN]].copy()
else:
    production_output = pd.DataFrame()

production_output["PredictedSalePrice"] = production_predictions

In [ ]:
def simulate_drift(production_df):
    drifted_df = production_df.copy()
    return drifted_df

drifted_production_df = simulate_drift(production_features)

In [16]:
drifted_production_df.to_csv("production_data_without_simulated_drift.csv",index=False)


In [17]:
# population stability index:

def calculate_psi(expected, actual, buckets=10):
    expected = pd.Series(expected).dropna()
    actual = pd.Series(actual).dropna()

    if expected.nunique() <= 1 or actual.nunique() <= 1:
        return 0.0

    breakpoints = np.percentile(expected, np.linspace(0, 100, buckets + 1))
    breakpoints = np.unique(breakpoints)

    if len(breakpoints) < 3:
        return 0.0

    expected_counts = np.histogram(expected, bins=breakpoints)[0]
    actual_counts = np.histogram(actual, bins=breakpoints)[0]

    expected_percent = expected_counts / max(len(expected), 1)
    actual_percent = actual_counts / max(len(actual), 1)

    expected_percent = np.where(expected_percent == 0, 0.0001, expected_percent)
    actual_percent = np.where(actual_percent == 0, 0.0001, actual_percent)

    psi_values = (actual_percent - expected_percent) * np.log(actual_percent / expected_percent)
    return float(np.sum(psi_values))

In [19]:
# To detect Drift
def detect_drift(reference_df, current_df):
    numeric_columns = reference_df.select_dtypes(include=["int64", "float64"]).columns
    numeric_columns = [column for column in numeric_columns if column != ID_COLUMN]
    drift_rows = []

    for column in numeric_columns:
        if column not in current_df.columns:
            continue

        reference_values = reference_df[column].dropna()
        current_values = current_df[column].dropna()

        if len(reference_values) == 0 or len(current_values) == 0:
            continue

        ks_statistic, ks_p_value = ks_2samp(reference_values, current_values)
        psi = calculate_psi(reference_values, current_values)

        alert = (ks_p_value < 0.05) or (psi >= 0.20)

        drift_rows.append(
            {
                "feature": column,
                "ks_statistic": round(float(ks_statistic), 4),
                "ks_p_value": round(float(ks_p_value), 6),
                "psi": round(float(psi), 4),
                "alert": alert,
            }
        )

    return pd.DataFrame(drift_rows).sort_values(["alert", "psi"], ascending=[False, False])


drift_report = detect_drift(X_train, drifted_production_df)
drift_report.to_csv("drift_report_new.csv", index=False)

In [20]:
drift_report.head(10)


,feature,ks_statistic,ks_p_value,psi,alert
15,GrLivArea,0.0585,0.022304,0.0244,True
22,TotRmsAbvGrd,0.0549,0.038418,0.0168,True
34,MoSold,0.0492,0.083244,0.0278,False
11,TotalBsmtSF,0.0407,0.224059,0.0211,False
7,MasVnrArea,0.0324,0.496152,0.0176,False
2,LotArea,0.0491,0.084419,0.0157,False
13,2ndFlrSF,0.0502,0.073001,0.0153,False
10,BsmtUnfSF,0.0276,0.694430,0.0137,False
25,GarageCars,0.0376,0.307932,0.0125,False
12,1stFlrSF,0.0304,0.573376,0.0121,False


In [21]:
# alerts :
def create_alerts(drift_report):
    alerts = []

    for _, row in drift_report.iterrows():
        if row["alert"]:
            alerts.append(
                f"ALERT: {row['feature']} drift detected "
                f"(KS p-value={row['ks_p_value']}, PSI={row['psi']})."
            )

    if not alerts:
        alerts.append("OK: No important drift detected.")

    return alerts


alerts = create_alerts(drift_report)

alerts_path = "alerts.txt"

with open(alerts_path, "w", encoding="utf-8") as f:
    f.write("\n".join(alerts))

for alert in alerts:
    print(alert)


ALERT: GrLivArea drift detected (KS p-value=0.022304, PSI=0.0244).
ALERT: TotRmsAbvGrd drift detected (KS p-value=0.038418, PSI=0.0168).
